In [1]:
#Imports 
import os
import sys
import pandas as pd
import numpy as np
import torch
from datasets import Dataset, DatasetDict
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding,
)
from sklearn.metrics import accuracy_score, precision_recall_fscore_support


d:\Deceptiscan\DECEPTISCAN---BROWSER-EXTENSION-main\Deceptiscan\extension\backend\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
class Config:
    epochs = 3
    batch_size = 8
    learning_rate = 2e-5
    max_length = 128
    dry_run = False  # Set to True if you want to test the notebook quickly with 100 samples

def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    if isinstance(predictions, tuple):
        predictions = predictions[0]
    
    preds = np.argmax(predictions, axis=1)
    precision, recall, f1, _ = precision_recall_fscore_support(labels, preds, average="binary")
    acc = accuracy_score(labels, preds)
    return {
        "accuracy": acc,
        "f1": f1,
        "precision": precision,
        "recall": recall
    }


In [3]:
config = Config()
current_dir = os.getcwd()

# Target your local datasets
dataset_dir = os.path.abspath(os.path.join(current_dir, "..", "dataset"))
train_path = os.path.join(dataset_dir, "train.csv")
test_path = os.path.join(dataset_dir, "test.csv")

# Target where fine-tuned model weights will be saved
model_save_dir = os.path.abspath(os.path.join(current_dir, "..", "models", "deberta-v3-detector"))


In [5]:
#load and prepare dataset
print(f"Loading datasets...")
if not os.path.exists(train_path) or not os.path.exists(test_path):
    raise FileNotFoundError(f"Preprocessed datasets not found in {dataset_dir}")

train_df = pd.read_csv(train_path)
test_df = pd.read_csv(test_path)

# Ensure correct columns and cast reviews to string to prevent tokenization crashes
train_df = train_df.dropna(subset=["text", "label"])
test_df = test_df.dropna(subset=["text", "label"])

train_df["text"] = train_df["text"].astype(str)
test_df["text"] = test_df["text"].astype(str)

train_df["label"] = train_df["label"].astype(int)
test_df["label"] = test_df["label"].astype(int)

if config.dry_run:
    print("--- RUNNING IN DRY RUN MODE ---")
    train_df = train_df.head(100)
    test_df = test_df.head(50)
    config.epochs = 1
    config.batch_size = 2
    print(f"Sampled {len(train_df)} training and {len(test_df)} testing rows.")

# Convert to Hugging Face datasets
train_dataset = Dataset.from_pandas(train_df)
test_dataset = Dataset.from_pandas(test_df)
dataset = DatasetDict({"train": train_dataset, "test": test_dataset})
print("Datasets loaded successfully!")


Loading datasets...
Datasets loaded successfully!


In [7]:
#load DeBERTa-v3 model
print("Loading tokenizer and pre-trained model (microsoft/deberta-v3-base)...")
tokenizer = AutoTokenizer.from_pretrained("microsoft/deberta-v3-base")
model = AutoModelForSequenceClassification.from_pretrained("microsoft/deberta-v3-base", num_labels=2, weights_only=False)


# Enable CUDA if GPU is available
device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)
print(f"Using device: {device}")


Loading tokenizer and pre-trained model (microsoft/deberta-v3-base)...


Some weights of DebertaV2ForSequenceClassification were not initialized from the model checkpoint at microsoft/deberta-v3-base and are newly initialized: ['classifier.bias', 'classifier.weight', 'pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Using device: cuda


In [8]:
#Tokenize datasets
def tokenize_function(examples):
    return tokenizer(examples["text"], padding=False, truncation=True, max_length=config.max_length)

print("Tokenizing datasets...")
tokenized_datasets = dataset.map(tokenize_function, batched=True, remove_columns=["text"])
print("Tokenization complete!")


Tokenizing datasets...


Map: 100%|██████████| 14332/14332 [00:00<00:00, 24949.16 examples/s]

Tokenization complete!


In [9]:
# Set up Data Collator for dynamic padding
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

os.makedirs(model_save_dir, exist_ok=True)
logs_dir = os.path.join(current_dir, "logs")
os.makedirs(logs_dir, exist_ok=True)

print("Configuring training arguments...")
training_args = TrainingArguments(
    output_dir=os.path.join(current_dir, "results"),
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=config.learning_rate,
    per_device_train_batch_size=config.batch_size,
    per_device_eval_batch_size=config.batch_size,
    num_train_epochs=config.epochs,
    weight_decay=0.01,
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    logging_dir=logs_dir,
    logging_steps=10 if config.dry_run else 100,
    fp16=torch.cuda.is_available(),           # Enables GPU mixed-precision training
    save_total_limit=1,
    report_to="none"
)

print("Initializing Trainer...")
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["test"],
    processing_class=tokenizer,               # Corrected for Transformers v5
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)


Configuring training arguments...
Initializing Trainer...


In [10]:
##TRAINING
print("Starting training...")
trainer.train()

print(f"Evaluating model on test dataset...")
metrics = trainer.evaluate()
print("Test metrics:")
for key, val in metrics.items():
    print(f"  {key}: {val}")

print(f"Saving model weights and tokenizer configurations to: {model_save_dir}...")
trainer.save_model(model_save_dir)
tokenizer.save_pretrained(model_save_dir)
print("Model saved successfully!")


The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 2, 'bos_token_id': 1}.


Starting training...


Epoch,Training Loss,Validation Loss,Accuracy,F1,Precision,Recall
1,0.351900,0.332251,0.863801,0.887906,0.859191,0.918607
2,0.291300,0.390862,0.867360,0.892179,0.853500,0.934529
3,0.165200,0.558253,0.863034,0.889052,0.847796,0.934529


Evaluating model on test dataset...


Test metrics:
  eval_loss: 0.3908621370792389
  eval_accuracy: 0.8673597543957577
  eval_f1: 0.8921785491463898
  eval_precision: 0.8534997287032013
  eval_recall: 0.9345294676806084
  eval_runtime: 36.9902
  eval_samples_per_second: 387.454
  eval_steps_per_second: 48.445
  epoch: 3.0
Saving model weights and tokenizer configurations to: d:\Deceptiscan\DECEPTISCAN---BROWSER-EXTENSION-main\Deceptiscan\extension\backend\models\deberta-v3-detector...
Model saved successfully!
